[Course material - Sustain.Brussels - "Avdanced AI workflows with LLM" - 20/04/2026 - 22/04/2026](https://github.com/Yannael/gen-ai-sustain-brussels-2604).

# Gradio Tutorial: Build an AI Web App

[Gradio](https://www.gradio.app/) is an open-source Python library that turns any Python function into a shareable web app in a few lines of code. Most demos on [Hugging Face Spaces](https://huggingface.co/spaces) are Gradio apps — open any Space and look at `app.py` to see the pattern.

**What you will build today — a 4-tab AI app:**

| Tab | Feature | Key concept |
|-----|---------|------------|
| 1 | Greeting | `gr.Interface` basics |
| 2 | Ask an LLM | Calling an external API |
| 3 | Image Generator | `gr.Image` output |
| 4 | Multi-turn Chat | `gr.ChatInterface` |

## Setup

Install the two packages you need. This takes about a minute.

In [ ]:
!pip install --quiet gradio huggingface_hub

---
## Part 1 — Hello World

Every Gradio app is built around three ingredients:
- `fn` — any Python function
- `inputs` — what goes in
- `outputs` — what comes out

Gradio builds the entire web UI from those three arguments.

In [ ]:
import gradio as gr

def greet(name):
    return "Hello, " + name + "!"

demo = gr.Interface(fn=greet, inputs="text", outputs="text")
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bbf866ae3db32e0a04.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Run the cell above and try typing your name in the app.

### Exercise 1 🛠️

Modify `greet` so it returns:
```
Hello, Alice! Your name has 5 letters.
```
Hint: use `len(name)` and an f-string.

In [ ]:
import gradio as gr

def greet(name):
    # TODO: modify this function
    return "Hello, " + name + "!" + " Your name has " + str(len(name)) + " letters."

gr.Interface(fn=greet, inputs="text", outputs="text").launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a13baf539cc53cc751.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## Part 2 — Customizing Components

Passing `"text"` is a shorthand. For more control, pass **component objects** directly.
You can also add a `title`, `description`, and `examples` to polish the UI.

In [ ]:
import gradio as gr

def greet(name, greeting):
    return f"{greeting}, {name}!"

demo = gr.Interface(
    fn=greet,
    inputs=[
        gr.Textbox(label="Your name", placeholder="e.g. Alice"),
        gr.Dropdown(["Hello", "Bonjour", "Hola", "Ciao"], label="Language", value="Hello"),
    ],
    outputs=gr.Textbox(label="Result"),
    title="Multilingual Greeter",
    description="Pick a language and enter your name!",
    examples=[["Alice", "Bonjour"], ["Carlos", "Hola"]],
)
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9af4badde154bdcd74.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Click one of the example rows at the bottom of the app.

> **Question:** What does clicking an example do? Where is this controlled in the code?

Other useful components: `gr.Slider`, `gr.Radio`, `gr.Checkbox`, `gr.Image`, `gr.File`...

Check other components at https://www.gradio.app/docs/gradio/interface


### Exercise 2 🛠️

Add a third input: `gr.Checkbox(label="Shout (ALL CAPS)")`.
When checked, return the greeting in uppercase.

In [ ]:
import gradio as gr

def greet(name, greeting, shout):
    result = f"{greeting}, {name}!"
    if shout:
        result = result.upper()
    return result

demo = gr.Interface(
    fn=greet,
    inputs=[
        gr.Textbox(label="Your name", placeholder="e.g. Alice"),
        gr.Dropdown(["Hello", "Bonjour", "Hola", "Ciao"], label="Language", value="Hello"),
        gr.Checkbox(label="Shout (ALL CAPS)"),
    ],
    outputs=gr.Textbox(label="Result"),
    title="Multilingual Greeter",
)
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7d928ea8553ce1424a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## Part 3 — Calling an LLM via the API

The function you pass to `gr.Interface` can do anything — including calling a remote AI model.
The Hugging Face `InferenceClient` gives you access to hundreds of models with one API key.

### Step 1 — Set up the client

In [ ]:
from huggingface_hub import InferenceClient
import os

os.environ["HF_API_KEY"] = "..."

HF_API_KEY = os.getenv("HF_API_KEY")

llm_client = InferenceClient(
    model="Qwen/Qwen2.5-7B-Instruct-1M:featherless-ai",
    api_key=HF_API_KEY,
)

> **Note:** In a real project, store API keys in environment variables or a `.env` file — never commit them to git.

### Step 2 — Write and test the function

In [ ]:
def ask_llm(prompt, max_tokens=200):
    response = llm_client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=int(max_tokens),
    )
    return response.choices[0].message.content

# Test outside of Gradio first
print(ask_llm("What is Gradio in one sentence?"))

Gradio is a simple, flexible library that allows you to create web interfaces for machine learning models and functions with minimal code.


### Step 3 — Wrap it in Gradio

In [ ]:
import gradio as gr

tab_llm = gr.Interface(
    fn=ask_llm,
    inputs=[
        gr.Textbox(label="Your question", placeholder="Ask me anything...", lines=3),
        gr.Slider(50, 500, value=200, step=50, label="Max tokens"),
    ],
    outputs=gr.Textbox(label="Answer", lines=8),
    title="Ask an LLM",
    description="Powered by Qwen via Hugging Face Inference API",
)
tab_llm.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b69115be84d72c9d6f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


> **Question:** Where is the computation happening — on your machine or on a remote server? How does the `max_tokens` slider affect the output?


### Exercise 3 🛠️

Let us add a third input: `gr.Textbox(label="System prompt", value="You are a helpful assistant.", lines=2)`.

Pass it as `{"role": "system", "content": system_prompt}` at the **start** of the messages list.

In [ ]:
import gradio as gr

def ask_llm_v2(prompt, system_prompt, max_tokens):
    response = llm_client.chat.completions.create(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
        max_tokens=int(max_tokens),
    )
    return response.choices[0].message.content

gr.Interface(
    fn=ask_llm_v2,
    inputs=[
        gr.Textbox(label="Your question", lines=3),
        gr.Textbox(label="System prompt", value="You are a helpful assistant.", lines=2),
        gr.Slider(50, 500, value=200, step=50, label="Max tokens"),
    ],
    outputs=gr.Textbox(label="Answer", lines=8),
    title="Ask an LLM (with system prompt)",
).launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e7431c3b887b405508.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## Part 4 — Generating Images

The same `InferenceClient` pattern works for image models.
The only difference: the output component is now `gr.Image` instead of `gr.Textbox`.

In [ ]:
from huggingface_hub import InferenceClient

img_client = InferenceClient(
    "black-forest-labs/FLUX.1-schnell",
    api_key=HF_API_KEY,
)

def generate_image(prompt):
    image = img_client.text_to_image(prompt)  # returns a PIL Image
    return image

In [ ]:
import gradio as gr

tab_image = gr.Interface(
    fn=generate_image,
    inputs=gr.Textbox(
        label="Describe your image",
        placeholder="A robot reading a book in a cozy library...",
        lines=2,
    ),
    outputs=gr.Image(label="Generated image"),
    title="AI Image Generator",
    description="Powered by FLUX.1-schnell",
)
tab_image.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://330b643602a8654ffd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### Exercise 4 🛠️

Add a `gr.Radio` input with options `["photorealistic", "watercolor painting", "pixel art", "pencil sketch"]`.

Before calling the API, append the style: `f"{prompt}, {style} style"`.

In [ ]:
import gradio as gr

def generate_image_v2(prompt, style):

    full_prompt = prompt+", "+style+" style"
    return img_client.text_to_image(full_prompt)

gr.Interface(
    fn=generate_image_v2,
    inputs=[
        gr.Textbox(label="Describe your image", lines=2),
        gr.Radio(
            ["photorealistic", "watercolor painting", "pixel art", "pencil sketch"],
            label="Style",
            value="photorealistic",
        ),
    ],
    outputs=gr.Image(label="Generated image"),
    title="AI Image Generator (with style)",
).launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d4d8d11a82ece9e7bf.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## Part 5 — Multi-turn Chat Interface

`gr.ChatInterface` is a higher-level component designed for conversations.

The key difference from `gr.Interface`: your function receives a `history` argument — the full list of past messages — so the model can remember context across turns.

In [ ]:
from huggingface_hub import InferenceClient
import gradio as gr

chat_client = InferenceClient(
    model="Qwen/Qwen2.5-7B-Instruct-1M:featherless-ai",
    api_key=HF_API_KEY,
)

def chat_fn(message, history):
    # history is already a list of {"role": ..., "content": ...} dicts
    messages = history + [{"role": "user", "content": message}]
    print(messages)
    response = chat_client.chat.completions.create(
        messages=messages,
    )
    return response.choices[0].message.content

tab_chat = gr.ChatInterface(
    fn=chat_fn,
    type="messages",
    title="Chat with Qwen",
    examples=["Tell me a joke", "Explain Gradio in 3 bullet points", "What is the capital of Belgium?"],
)
tab_chat.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://edc6bf645e516f347c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


> **Question:** Send a message, then ask "What did I just say?" — does the model remember?  
> Why does this work here but not in Part 3 (the single-turn interface)?

---
## Part 6 — Assembling the Final App

`gr.TabbedInterface` takes a list of interfaces and a list of tab names.  
Apart from the first tab, nothing else changes — we reuse the building blocks from each part above,  
incorporating the improvements from the exercises.

What does the first tab now do ?

In [ ]:
import gradio as gr
from huggingface_hub import InferenceClient

import os

os.environ["HF_API_KEY"] = ""

CHAT_MODEL = "Qwen/Qwen2.5-7B-Instruct-1M:featherless-ai"

HF_API_KEY = os.getenv("HF_API_KEY")

# ── Tab 1: Get weather  ────────────────────────────────────────────────────────────
def get_weather(location: str) -> str:
    """
    Retrieves the current weather for a given location.

    Args:
        location: The city or place for which to fetch the weather.
    """
    # In a real tool, call a weather API here.
    return f"The weather in {location} is sunny with mild temperatures."

tab1 = gr.Interface(
    fn=get_weather,
    inputs=[
        gr.Textbox(label="Location", placeholder="e.g. Brussels"),
    ],
    outputs=gr.Textbox(label="Weather"),
    title="Get Weather",
    examples=[["Brussels"], ["London"]],
)

# ── Tab 2: Ask an LLM ─────────────────────────────────────────────────────────
llm_client = InferenceClient(model=CHAT_MODEL, api_key=HF_API_KEY)

def ask_llm(prompt, system_prompt, max_tokens):
    response = llm_client.chat.completions.create(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
        max_tokens=int(max_tokens),
    )
    return response.choices[0].message.content

tab2 = gr.Interface(
    fn=ask_llm,
    inputs=[
        gr.Textbox(label="Your question", lines=3),
        gr.Textbox(label="System prompt", value="You are a helpful assistant.", lines=2),
        gr.Slider(50, 500, value=200, step=50, label="Max tokens"),
    ],
    outputs=gr.Textbox(label="Answer", lines=8),
    title="Ask an LLM",
)

# ── Tab 3: Image Generator ────────────────────────────────────────────────────
img_client = InferenceClient("black-forest-labs/FLUX.1-schnell", api_key=HF_API_KEY)

def generate_image(prompt, style):
    """
    Generate an image from a text description using Imagen via Google GenAI.

    Parameters
    ----------
    prompt : str
        A natural-language description of the image to generate.

    Returns
    -------
    PIL.Image.Image
        The generated image as a Pillow Image object, ready to display in Gradio.
    """

    full_prompt = f"{prompt}, {style} style" if style else prompt
    return img_client.text_to_image(full_prompt)

tab3 = gr.Interface(
    fn=generate_image,
    inputs=[
        gr.Textbox(label="Describe your image", placeholder="A futuristic city at sunset...", lines=2),
        gr.Radio(
            ["photorealistic", "watercolor painting", "pixel art", "pencil sketch"],
            label="Style",
            value="photorealistic",
        ),
    ],
    outputs=gr.Image(label="Generated image"),
    title="Image Generator",
)

# ── Tab 4: Multi-turn Chat ────────────────────────────────────────────────────
chat_client = InferenceClient(model=CHAT_MODEL, api_key=HF_API_KEY)

def chat_fn(message, history):
    messages = history + [{"role": "user", "content": message}]
    response = chat_client.chat.completions.create(messages=messages)
    return response.choices[0].message.content

tab4 = gr.ChatInterface(
    fn=chat_fn,
    title="Chat with Qwen",
    examples=["Tell me a joke", "Explain AI in simple terms", "What is Gradio?"],
)

# ── Assemble ──────────────────────────────────────────────────────────────────
app = gr.TabbedInterface(
    [tab1, tab2, tab3, tab4],
    tab_names=["Get weather", "Ask LLM", "Image Gen", "Chat"],
)
app.launch(mcp_server=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(
/usr/local/lib/python3.12/dist-packages/gradio/mcp.py:207: UserWarning: This MCP server includes a tool that has a gr.State input, which will not be updated between tool calls. The original, default value of the State will be used each time.
  warnings.warn(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ad8f85d479142b9fa1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)

🔨 Launching MCP server:
** Streamable HTTP URL: https://ad8f85d479142b9fa1.gradio.live/gradio_api/mcp/
* [Deprecated] SSE URL: https://ad8f85d479142b9fa1.gradio.live/gradio_api/mcp/sse


Congratulations — you have built a full AI-powered multi-tab web app in 30 minutes!



---
## Gradio Cheat Sheet

| Task | Code |
|------|------|
| Minimal app | `gr.Interface(fn=f, inputs="text", outputs="text").launch()` |
| Custom component | `gr.Textbox(label="...", placeholder="...", lines=2)` |
| Multiple inputs | `inputs=[comp1, comp2, ...]` |
| Image output | `outputs=gr.Image()` |
| Tabbed app | `gr.TabbedInterface([tab1, tab2], tab_names=["A", "B"])` |
| Multi-turn chat | `gr.ChatInterface(fn=f, type="messages")` |
